# Getting Historical Chain Constituents with LSEG Tick History REST API and LSEG Data Library for Python

The [LSEG Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python) `Chain` object returns a chain's constituents, which you can then use with `get_data` and `get_history`. But some bonds chains change its constituent over time because exchanges periodically update which bonds meet the  maturity and coupon criteria for delivery. For instance, newly issued Treasuries are added while older bonds that have fallen below the minimum remaining maturity are removed. 

Examples:
- **0#TYc1=DLV**: CBoT 10-Year US Treasury Note
- **0#FGBMc1=DLV**: Eurex Euro-Bobl (5-year German Govt Bond)
- **0#JGBc1=DLV**: Osaka Exchange 10-Year Japanese Government Bond

See [README.md](./README.md) for more about these chain rules.

The `Chain` object returns **current constituents** because it gets data from the Real-Time platform. To resolve constituents for a specific time, use the [LSEG Tick History REST API](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api), which supports current and past constituents, including inactive, historical-only instruments. 

Once you get the constituents of that specific time from the Tick History, you can then retrieve their historical data with Tick History, the Data Library, or another LSEG API such as [Data Platform APIs (RDP)](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis).

**Note**

You can also use the Tick History REST API for the entire workflow. See the [Tick History REST API tutorials](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorials-introduction) for details. This example is for developers who use the Data Library as their primary API.

In [1]:
import os
import time
import requests

## Prerequisite  

The Tick History - REST API requires the DataScope Select (DSS) credential and permission. Please contact your LSEG Account Manager or CSM team to help you get the DSS access credential for the Tick History API.

In [2]:
BASE_URL = "https://selectapi.datascope.lseg.com/RestApi/v1"
TIMEOUT = 30  # seconds

DSS_USERNAME = os.environ.get("DSS_USERNAME")
DSS_PASSWORD = os.environ.get("DSS_PASSWORD")

if not DSS_USERNAME or not DSS_PASSWORD:
    raise ValueError("Please set DSS_USERNAME and DSS_PASSWORD environment variables.")

## Authentication with DSS Server

First, request an authentication token from the DSS server. The request must include valid DSS credentials. For details, see [Tick History - REST API Tutorial 1: Connecting to the server](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-1-connecting-to-the-server).

The DSS authentication token is valid for 24 hours. Include it in the HTTP `Authorization` header as `Token {DSS Token}` in all Tick History data extraction requests.

This example uses a [Requests Session object](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) for all HTTP requests to DSS. A session reuses the TCP connection when making multiple requests to the same host, which can significantly improve performance. It also lets you set parameters, such as HTTP headers and cookies, that persist across requests.

In [ ]:

# Reuse this session and its headers for Tick History requests.
session = requests.Session()
session.headers["Content-Type"] = "application/json"
session.headers["Prefer"] = "respond-async"

# Get an access token using the DSS credentials.
try:
    auth_response = session.post(
        f"{BASE_URL}/Authentication/RequestToken",
        json={"Credentials": {"Username": DSS_USERNAME, "Password": DSS_PASSWORD}},
        timeout=TIMEOUT,
    )
    auth_response.raise_for_status()
    print(f"Authentication Tick History response: {auth_response.status_code}")
    token = auth_response.json()["value"]
    if token:
        print("Token obtained")
except requests.exceptions.RequestException as exc:
    raise RuntimeError("DSS authentication request failed") from exc

# Include the token in later API requests.
if token:
    session.headers["Authorization"] = f"Token {token}"

Authentication Tick History response: 200
Token obtained


## Resolving Historical Chain Constituents

The Tick History REST API's `/Search/HistoricalChainResolution` endpoint resolves the constituents of a Chain RIC for a specified date range. It supports currently active, or inactive 'historical only' instruments.

Notice that the `_session.post()` call does not include an `Authorization` header. We added the header to the `session` object during authentication, so Requests includes it automatically with every request made through that session. 

I define the historical chain constituent extraction in a `resolve_chain_request` function so we can call the endpoint multiple times throughout the workflow. Pass the `session` object to the function as `_session` each time.

You can find more detail regarding the `/Search/HistoricalChainResolution` endpoint from [Tick History - REST API Tutorial 15: Historical Chain Constituents Search](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-15-historical-chain-constituents-search).

In [ ]:

def resolve_chain_request(request_session, base_url, chain_ric, start, end, timeout):
    """Return the constituents of `chain_ric` between `start` and `end`."""
    print(f"Resolving {chain_ric} from {start} to {end}")

    payload = {
        "Request": {
            "ChainRics": [chain_ric],
            "Range": {"Start": start, "End": end},
        }
    }

    try:
        historical_chain_response = request_session.post(
            f"{base_url}/Search/HistoricalChainResolution",
            json=payload,
            timeout=timeout,
        )

        # HTTP 202 = still processing; poll the monitor URL in the Location header until it returns 200
        while historical_chain_response.status_code == 202:
            print("Request is still processing, waiting 5 seconds...")
            time.sleep(5)
            historical_chain_response = request_session.get(historical_chain_response.headers["Location"], timeout=timeout)

        # Stop here if the API reports an error.
        historical_chain_response.raise_for_status()
        print(f"Historical Chain Resolution response: {historical_chain_response.status_code}")
        # Only a completed 200 response contains the constituent data.
        if historical_chain_response.status_code == 200:
            constituents = historical_chain_response.json()["value"][0]["Constituents"]
            print(f"Tick History:  {len(constituents)} constituents returned")
            return constituents
    except requests.exceptions.RequestException as exc:
        raise RuntimeError("Historical chain resolution request failed") from exc


Lets make a first `/Search/HistoricalChainResolution` endpoint call via `resolve_chain_request` method. I am requesting *0#TYc1=DLV* (CBoT 10-Year US Treasury Note) RIC on the May and June 2024 period.

In [ ]:
CHAIN_RIC = "0#TYc1=DLV"
START_2024 = "2024-05-01T00:00:00.000Z"
END_2024 = "2024-06-30T00:00:00.000Z"

# Fetch 2024 constituents via resolve_chain_request()
constituents_2024 = resolve_chain_request(
    request_session=session,
    base_url=BASE_URL,
    chain_ric=CHAIN_RIC,
    start=START_2024,
    end=END_2024,
    timeout=TIMEOUT
)

Resolving 0#TYc1=DLV from 2024-05-01T00:00:00.000Z to 2024-06-30T00:00:00.000Z
Historical Chain Resolution response: 200
Tick History:  18 constituents returned


Once we got the response message that contains constituent RICs of that time, the code filter only an active RICs (`"Status": "Valid"`) to a `rics_2024` list variable.

In [ ]:
# Keep only valid RIC identifiers from the 2024 chain response.
rics_2024 = [
    item["Identifier"]
    for item in constituents_2024
    if item["Status"] == "Valid" and item["IdentifierType"] == "Ric"
]
rics_2024

['91282CBL4=',
 '91282CCB5=',
 '91282CCS8=',
 '91282CDJ7=',
 '91282CDY4=',
 '91282CEP2=',
 '91282CFF3=',
 '91282CFV8=',
 '91282CGM7=',
 '91282CHC8=',
 '91282CJQ5=',
 '91282CJX0=',
 '91282CKC4=',
 '91282CKF7=',
 '91282CKN0=',
 '91282CKQ3=',
 '91282CKU4=',
 '91282CKW0=']

A quick reminder: You can use the LSEG Tick History REST API for the entire workflow, including retrieving historical data for these constituents (`rics_2024`), without using any other LSEG APIs. For example:
- [REST API Tutorial 6: On Demand intraday bars extraction](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-6-on-demand-intraday-bars-extraction)
- [REST API Tutorial 4: On Demand tick data extraction](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-4-on-demand-tick-data-extraction)

The workflow below uses the LSEG Data Library with a Desktop Session to retrieve historical data for the same RICs and date range from Workspace. You can also use a Platform Session to retrieve data from the Data Platform ([see Data Library for Python QuickStart page](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python/quick-start)).

In [7]:
import lseg.data as ld 

In [8]:
ld.open_session()

<lseg.data.session.Definition object at 0x1dc84068830 {name='workspace'}>

I am demonstrating with Data Library `get_history` method to request historical data in a simplest way. You can see the example and full potential of this method from [EX-1.01.02-GetHistory.ipynb](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python/blob/lseg-data-examples/Examples/1-Access/EX-1.01.02-GetHistory.ipynb) example code.

In [11]:
constituents_2024 = ld.get_history(universe=rics_2024[0:6], fields=["BID","ASK"], interval="weekly" , start=START_2024, end= END_2024)

constituents_2024.head(20)

91282CBL4=            91282CCB5=            91282CCS8=             \
                  BID        ASK        BID        ASK        BID        ASK   
Date                                                                           
2024-05-03   80.78125     80.875  82.984375    83.0625   80.15625  80.210938   
2024-05-10  80.742188    80.8125  82.929688       83.0   80.09375  80.164062   
2024-05-17  81.164062      81.25  83.367188    83.4375    80.5625     80.625   
2024-05-24   80.84375  80.914062  83.054688  83.117188      80.25  80.304688   
2024-05-31  80.773438   80.84375  82.960938  83.023438  80.148438   80.21875   
2024-06-07   81.15625   81.21875  83.335938  83.398438  80.554688  80.609375   
2024-06-14  82.382812  82.476562   84.59375  84.695312  81.804688  81.929688   
2024-06-21   82.21875   82.28125   84.40625   84.46875   81.59375   81.65625   
2024-06-28  81.648438  81.703125  83.804688  83.867188   81.03125  81.078125   
2024-07-05  82.367188  82.460938  84.554688  84.648438  81.789062     81.875   

           91282CDJ7=            91282CDY4=            91282CEP2=             
                  BID        ASK        BID        ASK        BID        ASK  
Date                                                                          
2024-05-03  80.273438  80.320312    82.9375       83.0     89.125  89.179688  
2024-05-10      80.25    80.3125   82.90625   82.96875    89.0625  89.132812  
2024-05-17  80.742188  80.789062   83.40625   83.46875  89.601562  89.648438  
2024-05-24    80.4375  80.492188  83.101562   83.15625  89.273438  89.320312  
2024-05-31  80.320312     80.375  82.960938  83.015625   89.09375  89.148438  
2024-06-07   80.71875   80.78125   83.40625  83.445312  89.539062   89.59375  
2024-06-14       82.0     82.125  84.726562  84.820312  90.945312  91.039062  
2024-06-21    81.8125     81.875  84.539062   84.59375      90.75    90.8125  
2024-06-28  81.203125      81.25    83.8125     83.875   89.96875   90.03125  
2024-07-05  81.992188  82.046875  84.648438  84.710938    90.8125     90.875

Lets try to request the same *0#TYc1=DLV* constituent RICs on the May and June 2026 period.

In [ ]:
START_2026 = "2026-05-01T00:00:00.000Z"
END_2026   = "2026-06-30T00:00:00.000Z"

# Fetch 2026 constituents via resolve_chain_request()
constituents_2026 = resolve_chain_request(
    request_session = session, 
    base_url = BASE_URL, 
    chain_ric = CHAIN_RIC,
    start = START_2026, 
    end = END_2026, 
    timeout = TIMEOUT
)

Resolving 0#TYc1=DLV from 2026-05-01T00:00:00.000Z to 2026-06-30T00:00:00.000Z
Historical Chain Resolution response: 200
Tick History:  19 constituents returned


In [ ]:
# Keep only valid RIC identifiers from the 2026 chain response.
rics_2026 = [
    item["Identifier"]
    for item in constituents_2026
    if item["Status"] == "Valid" and item["IdentifierType"] == "Ric"
]
rics_2026

['91282CDJ7=',
 '91282CDY4=',
 '91282CFF3=',
 '91282CFV8=',
 '91282CGM7=',
 '91282CHC8=',
 '91282CHT1=',
 '91282CJJ1=',
 '91282CJZ5=',
 '91282CKQ3=',
 '91282CLF6=',
 '91282CPQ8=',
 '91282CPY1=',
 '91282CQC8=',
 '91282CQF1=',
 '91282CQN4=',
 '91282CQQ7=',
 '91282CQT1=',
 '91282CQW4=']

### Comparing 2024 and 2026 Chain Constituents

Now let's compare `rics_2024` and `rics_2026` side by side in a `pandas` DataFrame. Because the deliverable bond basket for *0#TYc1=DLV* updates over time—older Treasury notes drop out once they fall below the minimum remaining maturity requirement while newly issued notes are added—the constituent lists for May–June 2024 and May–June 2026 are different.

In [21]:
import pandas as pd

# Convert both RIC lists to sets so we can compare membership directly.
set_2024 = set(rics_2024)
set_2026 = set(rics_2026)

# Check whether both periods have the exact same constituents.
print(f"Same constituents in 2024 and 2026? {set_2024 == set_2026}")

# Place removed, unchanged, and added RICs side by side, padding shorter columns with "".
comparison_df = pd.concat(
    {
        "Removed (2024 only)": pd.Series(sorted(set_2024 - set_2026)),  # In 2024 but not 2026
        "Unchanged (both)": pd.Series(sorted(set_2024 & set_2026)),     # Present in both periods
        "Added (2026 only)": pd.Series(sorted(set_2026 - set_2024)),    # New in 2026
    },
    axis=1,
).fillna("")

comparison_df

Same constituents in 2024 and 2026? False


,Removed (2024 only),Unchanged (both),Added (2026 only)
0,91282CBL4=,91282CDJ7=,91282CHT1=
1,91282CCB5=,91282CDY4=,91282CJJ1=
2,91282CCS8=,91282CFF3=,91282CJZ5=
3,91282CEP2=,91282CFV8=,91282CLF6=
4,91282CJQ5=,91282CGM7=,91282CPQ8=
5,91282CJX0=,91282CHC8=,91282CPY1=
6,91282CKC4=,91282CKQ3=,91282CQC8=
7,91282CKF7=,,91282CQF1=
8,91282CKN0=,,91282CQN4=
9,91282CKU4=,,91282CQQ7=


Now I am using the Data Library `get_history` method to request historical data for 2026's constituents using the same query as 2024.

In [23]:
bcom_2026_data = ld.get_history(
    universe=rics_2026[0:6], 
    fields=["BID","ASK"], 
    interval="weekly", 
    start=START_2026, 
    end= END_2026
)

bcom_2026_data.head(20)

91282CDJ7=            91282CDY4=            91282CFF3=             \
                  BID        ASK        BID        ASK        BID        ASK   
Date                                                                           
2026-05-08  86.757812    86.8125   88.65625   88.71875     92.375    92.4375   
2026-05-15    85.6875  85.726562  87.523438  87.570312     91.125  91.171875   
2026-05-22  85.796875  85.859375  87.648438   87.71875  91.273438  91.335938   
2026-05-29     86.375  86.429688      88.25  88.289062  91.914062  91.960938   
2026-06-05  85.867188   85.90625  87.695312  87.757812    91.3125     91.375   
2026-06-12  86.164062   86.21875   88.03125  88.070312  91.664062  91.710938   
2026-06-19   86.15625  86.210938  88.007812  88.054688   91.65625  91.710938   
2026-06-26  86.632812   86.71875  88.492188  88.570312  92.179688  92.242188   
2026-07-03      86.25    86.3125   88.09375  88.140625  91.679688  91.726562   

           91282CFV8=            91282CGM7=            91282CHC8=             
                  BID        ASK        BID        ASK        BID        ASK  
Date                                                                          
2026-05-08  99.820312     99.875  96.070312     96.125   95.09375  95.132812  
2026-05-15   98.46875       98.5    94.6875  94.734375  93.671875  93.710938  
2026-05-22     98.625    98.6875  94.867188   94.90625  93.835938  93.890625  
2026-05-29  99.289062   99.34375   95.53125  95.570312  94.554688  94.585938  
2026-06-05     98.625    98.6875     94.875  94.929688  93.898438    93.9375  
2026-06-12   98.96875  99.007812   95.21875  95.257812      94.25   94.28125  
2026-06-19  98.953125       99.0  95.242188   95.28125  94.257812    94.3125  
2026-06-26  99.476562  99.554688   95.78125  95.851562  94.835938  94.890625  
2026-07-03  98.914062   98.96875  95.210938  95.257812      94.25   94.28125

To finish, I'll close the sessions used by both APIs. I can close the Data Library session with `ld.close_session()`; in the next step, I'll invalidate the Tick History token.

In [24]:
ld.close_session()

An application can revoke a Tick History REST API token using the `/Authentication/InvalidateToken` endpoint. After a successful request, the token can no longer be used.

In [ ]:
try:
    # Ask Tick History to revoke the current token.
    logout_response = session.post(
        f"{BASE_URL}/Authentication/InvalidateToken",
        json={"Token": {"Token": token}},
        timeout=TIMEOUT,
    )
    print(f"Logout Tick History response: {logout_response.status_code}, {logout_response.text}")
    # Raise an error if the token could not be revoked.
    logout_response.raise_for_status()
except requests.exceptions.RequestException as exc:
    raise RuntimeError("DSS InvalidateToken request failed") from exc


Logout Tick History response: 200, {"@odata.context":"https://selectapi.datascope.lseg.com/RestApi/v1/$metadata#Edm.String","value":"Token has been invalidated successfully."}
